# Section A

# Q1

In [1]:
import random

In [24]:
#  QUESTION 1 - First-Choice & Stochastic Hill Climbing
#  Landscape: states 1..12, f(s) values below

LANDSCAPE = [4, 9, 6, 11, 8, 15, 10, 7, 13, 5, 16, 12]
          # s1  s2  s3  s4  s5  s6  s7  s8  s9 s10 s11 s12

In [25]:
# Helper Function: get neighbours of a state (left and right, 1-indexed)
def get_neighbours(state, landscape):
    neighbours = []
    idx = state - 1  # convert to 0-indexed
    if idx - 1 >= 0:
        neighbours.append(state - 1)   # left neighbour
    if idx + 1 < len(landscape):
        neighbours.append(state + 1)   # right neighbour
    return neighbours

In [26]:
#  PART (a) - Implement First-Choice HC and Stochastic HC
def first_choice_hc(landscape, start):
    """
    First-Choice Hill Climbing:
    Check left neighbour first, then right.
    Move immediately on the FIRST improvement found.
    """
    current = start
    path = [current]

    while True:
        neighbours = get_neighbours(current, landscape)
        moved = False
        for neighbour in neighbours:           # left is checked before right
            if landscape[neighbour - 1] > landscape[current - 1]:
                current = neighbour            # move immediately
                path.append(current)
                moved = True
                break                          # don't check further neighbours
        if not moved:
            break                              # no improvement found = terminate

    return path, current

In [27]:
def stochastic_hc(landscape, start):
    """
    Stochastic Hill Climbing:
    Collect ALL strictly uphill neighbours, then pick ONE randomly.
    """
    current = start
    path = [current]

    while True:
        neighbours = get_neighbours(current, landscape)
        uphill = [n for n in neighbours if landscape[n - 1] > landscape[current - 1]]

        if not uphill:
            break                              # no uphill neighbour = terminate

        current = random.choice(uphill)        # pick randomly from uphill moves
        path.append(current)

    return path, current

In [29]:
#  PART (a) - main() : run both from every starting state

def main_part_a():
    print("PART (a) - Running First-Choice HC and Stochastic HC from all states")
    print(f"\nLandscape: {LANDSCAPE}")
    print(f"States 1-12, f(s) values shown above\n")

    print(f"{'Start':<7} {'Algorithm':<22} {'Path':<35} {'Terminal':<10} {'Steps'}")

    for start in range(1, 13):
        # First-Choice
        path_fc, term_fc = first_choice_hc(LANDSCAPE, start)
        print(f"{start:<7} {'First-Choice HC':<22} {str(path_fc):<35} {term_fc:<10} {len(path_fc)-1}")

        # Stochastic (fix seed per start for reproducibility in part a)
        random.seed(42 + start)
        path_st, term_st = stochastic_hc(LANDSCAPE, start)
        print(f"{start:<7} {'Stochastic HC':<22} {str(path_st):<35} {term_st:<10} {len(path_st)-1}")
        print()

In [30]:
main_part_a()

PART (a) - Running First-Choice HC and Stochastic HC from all states

Landscape: [4, 9, 6, 11, 8, 15, 10, 7, 13, 5, 16, 12]
States 1-12, f(s) values shown above

Start   Algorithm              Path                                Terminal   Steps
1       First-Choice HC        [1, 2]                              2          1
1       Stochastic HC          [1, 2]                              2          1

2       First-Choice HC        [2]                                 2          0
2       Stochastic HC          [2]                                 2          0

3       First-Choice HC        [3, 2]                              2          1
3       Stochastic HC          [3, 4]                              4          1

4       First-Choice HC        [4]                                 4          0
4       Stochastic HC          [4]                                 4          0

5       First-Choice HC        [5, 4]                              4          1
5       Stochastic HC         

In [72]:
#  PART (b) - Analysis + Run Stochastic HC 50 times

def main_part_b():
    print("PART (b) - Analysis + Run Stochastic HC 50 times")

    # How many starting states reach the global max (state 11, f=16)? ---
    print("\n[b.1] How many starting states reach global maximum (state 11, f=16)?")

    fc_reach_11 = 0
    st_reach_11 = 0

    print(f"{'Start':<7} {'FC Terminal':<15} {'FC->11?':<10} {'ST Terminal':<15} {'ST->11?'}")

    for start in range(1, 13):
        path_fc, term_fc = first_choice_hc(LANDSCAPE, start)
        random.seed(42 + start)
        path_st, term_st = stochastic_hc(LANDSCAPE, start)

        fc_yes = "YES" if term_fc == 11 else "no"
        st_yes = "YES" if term_st == 11 else "no"

        if term_fc == 11: fc_reach_11 += 1
        if term_st == 11: st_reach_11 += 1

        print(f"{start:<7} {term_fc:<15} {fc_yes:<10} {term_st:<15} {st_yes}")

    print(f"\nSummary: First-Choice reaches state 11 from {fc_reach_11}/12 starting states")
    print(f"         Stochastic HC reaches state 11 from {st_reach_11}/12 starting states")

    # Starting states where FC and Stochastic reach DIFFERENT terminals ---
    print("\n[b.2] Starting states where FC and Stochastic reach DIFFERENT terminals")

    diverge_found = False
    for start in range(1, 13):
        path_fc, term_fc = first_choice_hc(LANDSCAPE, start)
        random.seed(42 + start)
        path_st, term_st = stochastic_hc(LANDSCAPE, start)

        if term_fc != term_st:
            diverge_found = True
            print(f"\nStart = {start}")
            print(f"  First-Choice path : {path_fc}  -> terminal state {term_fc} (f={LANDSCAPE[term_fc-1]})")
            print(f"  Stochastic path   : {path_st}  -> terminal state {term_st} (f={LANDSCAPE[term_st-1]})")
            print(f"  Reason: At some step, FC always picks LEFT neighbour first (if it is")
            print(f"  better), while Stochastic collects ALL uphill neighbours and picks")
            print(f"  randomly. This can lead them to different peaks.")

    if not diverge_found:
        print("No divergence found with this seed. Both algorithms reach the same terminals.")



    # Run Stochastic HC 50 times from s=4
    print("\n[b.3] Stochastic HC run 50 times from start = 4 (no fixed seed)")

    reach_11 = 0
    other_terminals = {}

    for _ in range(50):
        # No fixed seed, true randomness
        path_st, term_st = stochastic_hc(LANDSCAPE, 4)
        if term_st == 11:
            reach_11 += 1
        else:
            other_terminals[term_st] = other_terminals.get(term_st, 0) + 1

    percentage = (reach_11 / 50) * 100
    print(f"  Reached state 11 (global max): {reach_11}/50 times = {percentage:.1f}%")
    print(f"  Other terminals reached: {other_terminals}")


In [73]:
main_part_b()

PART (b) - Analysis + Run Stochastic HC 50 times

[b.1] How many starting states reach global maximum (state 11, f=16)?
Start   FC Terminal     FC->11?    ST Terminal     ST->11?
1       2               no         2               no
2       2               no         2               no
3       2               no         4               no
4       4               no         4               no
5       4               no         4               no
6       4               no         7               no
7       7               no         7               no
8       7               no         7               no
9       7               no         7               no
10      7               no         11              YES
11      11              YES        11              YES
12      11              YES        11              YES

Summary: First-Choice reaches state 11 from 2/12 starting states
         Stochastic HC reaches state 11 from 3/12 starting states

[b.2] Starting states where FC and St

In [74]:
print("""
Explanation:
  - State 11 has f=16, which is the global maximum in the landscape.
  - First-Choice always takes the left neighbour if it is better, which can
    cause it to get trapped at local maxima (e.g., state 6 with f=15).
  - Stochastic HC picks randomly among uphill moves, so from the same start
    it may reach a different local maximum than First-Choice.
""")


Explanation:
  - State 11 has f=16, which is the global maximum in the landscape.
  - First-Choice always takes the left neighbour if it is better, which can
    cause it to get trapped at local maxima (e.g., state 6 with f=15).
  - Stochastic HC picks randomly among uphill moves, so from the same start
    it may reach a different local maximum than First-Choice.



In [75]:
print("""
Explanation:
  Divergence happens because First-Choice has a fixed bias (always checks
  left first), while Stochastic HC has randomness. From certain starts,
  First-Choice gets locked into a direction toward a local max, while
  Stochastic may randomly choose the path toward the global max.
""")


Explanation:
  Divergence happens because First-Choice has a fixed bias (always checks
  left first), while Stochastic HC has randomness. From certain starts,
  First-Choice gets locked into a direction toward a local max, while
  Stochastic may randomly choose the path toward the global max.



In [76]:
print(f"""
Explanation:
  From state 4 (f=11), both left neighbour s3 (f=6) and right neighbour
  s5 (f=8) are worse, so the algorithm terminates immediately at state 4
  in most runs. This reveals that Stochastic HC is unreliable when the
  starting state is already a local maximum - it cannot escape, regardless
  of how many times it is run. The randomness only helps when there are
  multiple uphill choices available.
""")


Explanation:
  From state 4 (f=11), both left neighbour s3 (f=6) and right neighbour
  s5 (f=8) are worse, so the algorithm terminates immediately at state 4
  in most runs. This reveals that Stochastic HC is unreliable when the
  starting state is already a local maximum - it cannot escape, regardless
  of how many times it is run. The randomness only helps when there are
  multiple uphill choices available.



In [77]:
#  PART (c) - Plateau Handling + Sideways Move

# Modified landscape: states 5, 6, 7 all get f = 15
LANDSCAPE_PLATEAU = [4, 9, 6, 11, 15, 15, 15, 7, 13, 5, 16, 12]
                  # s1  s2  s3  s4  s5  s6  s7  s8  s9 s10 s11 s12

In [78]:
def first_choice_hc_plateau(landscape, start):
    """
    First-Choice HC with PLATEAU DETECTION.
    Prints a warning when stuck on a plateau (equal neighbour, no better).
    """
    current = start
    path = [current]
    plateau_hit = False

    while True:
        neighbours = get_neighbours(current, landscape)
        moved = False

        # Check for strictly better first
        for neighbour in neighbours:
            if landscape[neighbour - 1] > landscape[current - 1]:
                current = neighbour
                path.append(current)
                moved = True
                break

        if not moved:
            # Check if there is at least one equal neighbour
            equal = [n for n in neighbours if landscape[n - 1] == landscape[current - 1]]
            if equal:
                print(f"  [WARNING] Plateau detected at state {current} (f={landscape[current-1]}). No strictly better neighbour.")
                plateau_hit = True
            break

    return path, current, plateau_hit

In [79]:
def stochastic_hc_plateau(landscape, start):
    """
    Stochastic HC with PLATEAU DETECTION.
    """
    current = start
    path = [current]
    plateau_hit = False

    while True:
        neighbours = get_neighbours(current, landscape)
        uphill = [n for n in neighbours if landscape[n - 1] > landscape[current - 1]]

        if uphill:
            current = random.choice(uphill)
            path.append(current)
        else:
            equal = [n for n in neighbours if landscape[n - 1] == landscape[current - 1]]
            if equal:
                print(f"  [WARNING] Plateau detected at state {current} (f={landscape[current-1]}). No strictly better neighbour.")
                plateau_hit = True
            break

    return path, current, plateau_hit

In [80]:
def first_choice_hc_sideways(landscape, start, max_sideways=10):
    """
    First-Choice HC with SIDEWAYS MOVES (cap = 10).
    Allows moving to equal-valued neighbours, but limits sideways moves.
    """
    current = start
    path = [current]
    sideways_count = 0

    while True:
        neighbours = get_neighbours(current, landscape)
        moved = False

        # Try strictly better first
        for neighbour in neighbours:
            if landscape[neighbour - 1] > landscape[current - 1]:
                current = neighbour
                path.append(current)
                moved = True
                break

        # If no strictly better, try sideways (if cap not reached)
        # For First-Choice, prefer the HIGHER-indexed equal neighbour (rightward)
        # to avoid bouncing left and right on the plateau
        if not moved and sideways_count < max_sideways:
            equal = [n for n in neighbours if landscape[n - 1] == landscape[current - 1]]
            if equal:
                current = max(equal)           # pick rightward to cross plateau
                path.append(current)
                sideways_count += 1
                moved = True

        if not moved:
            break

    return path, current

In [81]:
def stochastic_hc_sideways(landscape, start, max_sideways=10):
    """
    Stochastic HC with SIDEWAYS MOVES (cap = 10).
    """
    current = start
    path = [current]
    sideways_count = 0

    while True:
        neighbours = get_neighbours(current, landscape)
        uphill = [n for n in neighbours if landscape[n - 1] > landscape[current - 1]]

        if uphill:
            current = random.choice(uphill)
            path.append(current)
        elif sideways_count < max_sideways:
            equal = [n for n in neighbours if landscape[n - 1] == landscape[current - 1]]
            if equal:
                current = random.choice(equal)
                path.append(current)
                sideways_count += 1
            else:
                break
        else:
            break

    return path, current

In [91]:
def main_part_c():
    print("PART (c) Plateau Landscape: states 5,6,7 all have f=15")
    print(f"\nModified Landscape: {LANDSCAPE_PLATEAU}")
    print("States 5, 6, 7 now all have f=15 (plateau region)\n")
    #  Without sideways moves
    print("[c.1] WITHOUT sideways moves - Plateau Detection")

    fc_plateau_stuck = 0
    st_plateau_stuck = 0

    print("\n First-Choice HC with plateau detection --")
    for start in range(1, 13):
        path, term, hit = first_choice_hc_plateau(LANDSCAPE_PLATEAU, start)
        if hit:
            fc_plateau_stuck += 1
        print(f"  Start={start}: path={path}, terminal={term} (f={LANDSCAPE_PLATEAU[term-1]}), plateau={'YES' if hit else 'no'}")

    print(f"\n  => First-Choice HC got stuck on plateau in {fc_plateau_stuck}/12 runs\n")

    print(" Stochastic HC with plateau detection --")
    for start in range(1, 13):
        random.seed(42 + start)
        path, term, hit = stochastic_hc_plateau(LANDSCAPE_PLATEAU, start)
        if hit:
            st_plateau_stuck += 1
        print(f"  Start={start}: path={path}, terminal={term} (f={LANDSCAPE_PLATEAU[term-1]}), plateau={'YES' if hit else 'no'}")

    print(f"\n  => Stochastic HC got stuck on plateau in {st_plateau_stuck}/12 runs")



    # With sideways moves
    print("\n[c.2] WITH sideways moves (cap = 10)")

    fc_success = 0   # reaches global max (state 11)
    st_success = 0

    print("\n-- First-Choice HC with sideways moves --")
    for start in range(1, 13):
        path, term = first_choice_hc_sideways(LANDSCAPE_PLATEAU, start)
        if term == 11:
            fc_success += 1
        print(f"  Start={start}: path={path}, terminal={term} (f={LANDSCAPE_PLATEAU[term-1]}), reached_11={'YES' if term==11 else 'no'}")

    print(f"\n  => First-Choice HC with sideways: reached global max in {fc_success}/12 runs\n")

    print(" Stochastic HC with sideways moves --")
    for start in range(1, 13):
        random.seed(42 + start)
        path, term = stochastic_hc_sideways(LANDSCAPE_PLATEAU, start)
        if term == 11:
            st_success += 1
        print(f"  Start={start}: path={path}, terminal={term} (f={LANDSCAPE_PLATEAU[term-1]}), reached_11={'YES' if term==11 else 'no'}")

    print(f"\n  => Stochastic HC with sideways: reached global max in {st_success}/12 runs")
    print(f"""
Summary comparison:
    Without sideways: FC stuck in {fc_plateau_stuck}/12, Stochastic stuck in {st_plateau_stuck}/12
    With sideways:    FC reached global max in {fc_success}/12, Stochastic in {st_success}/12""")

In [89]:
main_part_c()

PART (c) Plateau Landscape: states 5,6,7 all have f=15

Modified Landscape: [4, 9, 6, 11, 15, 15, 15, 7, 13, 5, 16, 12]
States 5, 6, 7 now all have f=15 (plateau region)

[c.1] WITHOUT sideways moves - Plateau Detection

 First-Choice HC with plateau detection --
  Start=1: path=[1, 2], terminal=2 (f=9), plateau=no
  Start=2: path=[2], terminal=2 (f=9), plateau=no
  Start=3: path=[3, 2], terminal=2 (f=9), plateau=no
  [WARNING] Plateau detected at state 5 (f=15). No strictly better neighbour.
  Start=4: path=[4, 5], terminal=5 (f=15), plateau=YES
  [WARNING] Plateau detected at state 5 (f=15). No strictly better neighbour.
  Start=5: path=[5], terminal=5 (f=15), plateau=YES
  [WARNING] Plateau detected at state 6 (f=15). No strictly better neighbour.
  Start=6: path=[6], terminal=6 (f=15), plateau=YES
  [WARNING] Plateau detected at state 7 (f=15). No strictly better neighbour.
  Start=7: path=[7], terminal=7 (f=15), plateau=YES
  [WARNING] Plateau detected at state 7 (f=15). No strict

In [92]:
print("""
Explanation (without sideways moves):
  When states 5, 6, 7 all have f=15, any algorithm reaching one of them
  sees equal-valued neighbours but no strictly better neighbour. Without
  sideways moves, the algorithm simply stops at the edge of the plateau,
  unable to cross it and reach the true global maximum at state 11 (f=16).
""")


Explanation (without sideways moves):
  When states 5, 6, 7 all have f=15, any algorithm reaching one of them
  sees equal-valued neighbours but no strictly better neighbour. Without
  sideways moves, the algorithm simply stops at the edge of the plateau,
  unable to cross it and reach the true global maximum at state 11 (f=16).



In [131]:
print(f"""
Explanation (with sideways moves - 4 to 5 sentences):
  The sideways-move extension allows both algorithms to traverse the plateau
  (states 5, 6, 7 with f=15) instead of stopping at its edge. First-Choice HC
  benefits significantly because it always picks the leftmost equal neighbour,
  which gives it a consistent direction to cross the plateau toward state 8
  and eventually reach state 11. Stochastic HC also benefits, but it picks a
  random equal neighbour, meaning it may wander on the plateau and waste
  sideways-move budget without making progress. The cap of 10 is important
  because without it, the algorithm could loop forever on a flat plateau
  (especially Stochastic HC, which picks randomly). The cap forces termination
  and prevents infinite loops, which is critical for reliability.
""")


Explanation (with sideways moves - 4 to 5 sentences):
  The sideways-move extension allows both algorithms to traverse the plateau
  (states 5, 6, 7 with f=15) instead of stopping at its edge. First-Choice HC
  benefits significantly because it always picks the leftmost equal neighbour,
  which gives it a consistent direction to cross the plateau toward state 8
  and eventually reach state 11. Stochastic HC also benefits, but it picks a
  random equal neighbour, meaning it may wander on the plateau and waste
  sideways-move budget without making progress. The cap of 10 is important
  because without it, the algorithm could loop forever on a flat plateau
  (especially Stochastic HC, which picks randomly). The cap forces termination
  and prevents infinite loops, which is critical for reliability.



In [94]:
#  RUN ALL PARTS
if __name__ == "__main__":
    main_part_a()
    main_part_b()
    main_part_c()

PART (a)  Random Restart HC Implementation

Landscape (14 states): [5, 8, 6, 12, 9, 7, 17, 14, 10, 6, 19, 15, 11, 8]
Global maximum: state 11, f(11) = 19

[a.1] Local Maxima in Q2 Landscape:
---------------------------------------------
  State  2 -> f(2) = 8
  State  4 -> f(4) = 12
  State  7 -> f(7) = 17
  State 11 -> f(11) = 19

  Total local maxima found: 4

[a.2] RRHC with num_restarts=20 -- First-Choice variant
Restart    Start    Terminal     f-value    Global Max Found?
-------------------------------------------------------
1          3        2            8          no
2          8        7            17         no
3          8        7            17         no
4          13       11           19         YES
5          3        2            8          no
6          12       11           19         YES
7          7        7            17         no
8          12       11           19         YES
9          6        4            12         no
10         7        7            17

#Q2

In [ ]:
import random

In [46]:
#  QUESTION 2 - Random Restart Hill Climbing
#  Warehouse robot scenario: 14 positions, multiple local optima

# Q2 Landscape: 14 states
LANDSCAPE = [5, 8, 6, 12, 9, 7, 17, 14, 10, 6, 19, 15, 11, 8]
          # s1  s2  s3  s4  s5  s6  s7  s8  s9 s10 s11 s12 s13 s14


# Global maximum: state 11, f(11) = 19

In [47]:
def get_neighbours(state, landscape):
    """Return left and right neighbours of a state (1-indexed)."""
    neighbours = []
    idx = state - 1
    if idx - 1 >= 0:
        neighbours.append(state - 1)
    if idx + 1 < len(landscape):
        neighbours.append(state + 1)
    return neighbours

In [48]:
def first_choice_hc(landscape, start):
    """
    First-Choice HC: check left neighbour first, move on first improvement.
    Returns (path, terminal_state).
    """
    current = start
    path = [current]
    while True:
        neighbours = get_neighbours(current, landscape)
        moved = False
        for neighbour in neighbours:
            if landscape[neighbour - 1] > landscape[current - 1]:
                current = neighbour
                path.append(current)
                moved = True
                break
        if not moved:
            break
    return path, current

In [49]:
def stochastic_hc(landscape, start):
    """
    Stochastic HC: collect all uphill neighbours, pick one randomly.
    Returns (path, terminal_state).
    """
    current = start
    path = [current]
    while True:
        neighbours = get_neighbours(current, landscape)
        uphill = [n for n in neighbours if landscape[n - 1] > landscape[current - 1]]
        if not uphill:
            break
        current = random.choice(uphill)
        path.append(current)
    return path, current

In [50]:
#  PART (a) - Random Restart HC + find_local_maxima

def find_local_maxima(landscape):
    """
    Returns all states (1-indexed) whose f-value is strictly greater
    than BOTH neighbours. Boundary states only need to beat one neighbour.
    """
    local_maxima = []
    n = len(landscape)
    for i in range(n):
        state = i + 1
        val = landscape[i]
        left_ok  = (i == 0)     or (val > landscape[i - 1])
        right_ok = (i == n - 1) or (val > landscape[i + 1])
        if left_ok and right_ok:
            local_maxima.append(state)
    return local_maxima

In [51]:
def random_restart_hc(landscape, num_restarts, variant='first_choice'):
    """
    Random Restart Hill Climbing.
    - Each restart: pick a random start, run HC, record result.
    - Returns (best_state, best_value, all_results).
    - all_results is a list of (start, terminal, path) tuples.
    - variant: 'first_choice' or 'stochastic'
    """
    best_state = None
    best_value = -1
    all_results = []

    for _ in range(num_restarts):
        # Pick start state uniformly at random (1-indexed)
        start = random.randint(0, len(landscape) - 1) + 1

        if variant == 'first_choice':
            path, terminal = first_choice_hc(landscape, start)
        else:
            path, terminal = stochastic_hc(landscape, start)

        terminal_value = landscape[terminal - 1]
        all_results.append((start, terminal, path))

        if terminal_value > best_value:
            best_value = terminal_value
            best_state = terminal

    return best_state, best_value, all_results

In [53]:
def main_part_a():
    print("PART (a)  Random Restart HC Implementation")

    print(f"\nLandscape (14 states): {LANDSCAPE}")
    print("Global maximum: state 11, f(11) = 19\n")

    # Find and print all local maxima
    local_maxima = find_local_maxima(LANDSCAPE)
    print("[a.1] Local Maxima in Q2 Landscape:")
    print("-" * 45)
    for s in local_maxima:
        print(f"  State {s:>2} -> f({s}) = {LANDSCAPE[s-1]}")
    print(f"\n  Total local maxima found: {len(local_maxima)}")


    # Run RRHC with 20 restarts under both variants
    print("\n[a.2] RRHC with num_restarts=20 -- First-Choice variant")
    random.seed(100)
    best_s, best_v, results_fc = random_restart_hc(LANDSCAPE, 20, variant='first_choice')

    global_max_state = 11   # f(11) = 19

    print(f"{'Restart':<10} {'Start':<8} {'Terminal':<12} {'f-value':<10} {'Global Max Found?'}")
    print("-" * 55)
    for i, (start, terminal, path) in enumerate(results_fc, 1):
        found = "YES" if terminal == global_max_state else "no"
        print(f"{i:<10} {start:<8} {terminal:<12} {LANDSCAPE[terminal-1]:<10} {found}")

    print(f"\n  Best state found: {best_s}  |  Best f-value: {best_v}")
    print(f"  Global max (state 11) found: {'YES' if best_s == global_max_state else 'NO'}")

    print("\n[a.3] RRHC with num_restarts=20 -- Stochastic variant")
    random.seed(200)
    best_s2, best_v2, results_st = random_restart_hc(LANDSCAPE, 20, variant='stochastic')

    print(f"{'Restart':<10} {'Start':<8} {'Terminal':<12} {'f-value':<10} {'Global Max Found?'}")
    for i, (start, terminal, path) in enumerate(results_st, 1):
        found = "YES" if terminal == global_max_state else "no"
        print(f"{i:<10} {start:<8} {terminal:<12} {LANDSCAPE[terminal-1]:<10} {found}")

    print(f"\n  Best state found: {best_s2}  |  Best f-value: {best_v2}")
    print(f"  Global max (state 11) found: {'YES' if best_s2 == global_max_state else 'NO'}")

In [55]:
main_part_a()

PART (a)  Random Restart HC Implementation

Landscape (14 states): [5, 8, 6, 12, 9, 7, 17, 14, 10, 6, 19, 15, 11, 8]
Global maximum: state 11, f(11) = 19

[a.1] Local Maxima in Q2 Landscape:
---------------------------------------------
  State  2 -> f(2) = 8
  State  4 -> f(4) = 12
  State  7 -> f(7) = 17
  State 11 -> f(11) = 19

  Total local maxima found: 4

[a.2] RRHC with num_restarts=20 -- First-Choice variant
Restart    Start    Terminal     f-value    Global Max Found?
-------------------------------------------------------
1          3        2            8          no
2          8        7            17         no
3          8        7            17         no
4          13       11           19         YES
5          3        2            8          no
6          12       11           19         YES
7          7        7            17         no
8          12       11           19         YES
9          6        4            12         no
10         7        7            17

In [56]:
print("""
Explanation:
  A local maximum is any state whose f-value is strictly greater than
  both its neighbours. In this 14-state landscape we have multiple local
  peaks, which is exactly why basic HC gets trapped -- it climbs to the
  nearest peak and stops, never seeing the true global max at state 11.
""")


Explanation:
  A local maximum is any state whose f-value is strictly greater than
  both its neighbours. In this 14-state landscape we have multiple local
  peaks, which is exactly why basic HC gets trapped -- it climbs to the
  nearest peak and stops, never seeing the true global max at state 11.



In [58]:
print("""
Explanation:
  Random Restart solves the local maximum problem by simply re-running HC
  from a fresh random start whenever it gets stuck. Over many restarts,
  the algorithm is likely to eventually start near the global maximum and
  climb to it. First-Choice is deterministic so the same start always gives
  the same terminal; Stochastic adds randomness within each run as well.
""")


Explanation:
  Random Restart solves the local maximum problem by simply re-running HC
  from a fresh random start whenever it gets stuck. Over many restarts,
  the algorithm is likely to eventually start near the global maximum and
  climb to it. First-Choice is deterministic so the same start always gives
  the same terminal; Stochastic adds randomness within each run as well.



In [61]:
#  PART (b) - Empirical vs Theoretical Probability Experiments

def main_part_b():
    print("PART (b) - Empirical vs Theoretical Probability of Finding Global Max")

    global_max_state = 11
    restart_counts = [1, 3, 5, 10, 20]

    # Step 1: Derive p from Q2 landscape using First-Choice HC
    # p = fraction of starting states that reach state 11 under First-Choice HC
    print("\n[b.1] Deriving p: fraction of starts reaching state 11 via First-Choice HC")
    print("-" * 60)

    reach_count = 0
    total_states = len(LANDSCAPE)

    print(f"{'Start':<8} {'Terminal':<12} {'Reached 11?'}")
    print("-" * 35)
    for start in range(1, total_states + 1):
        path, terminal = first_choice_hc(LANDSCAPE, start)
        reached = (terminal == global_max_state)
        if reached:
            reach_count += 1
        print(f"{start:<8} {terminal:<12} {'YES' if reached else 'no'}")

    p = reach_count / total_states
    print(f"\n  States reaching global max: {reach_count} / {total_states}")
    print(f"  p = {reach_count}/{total_states} = {p:.4f}")

    # Step 2: Empirical probabilities (100 independent trials each)
    print("\n[b.2] Empirical Probability: 100 independent trials per restart count n")

    empirical_probs = {}
    TRIALS = 100

    for n in restart_counts:
        success = 0
        for trial in range(TRIALS):
            # Each trial gets its own seed so they are independent
            random.seed(trial * 1000 + n)
            best_s, best_v, _ = random_restart_hc(LANDSCAPE, n, variant='first_choice')
            if best_s == global_max_state:
                success += 1
        prob = success / TRIALS
        empirical_probs[n] = prob

    print(f"{'n (restarts)':<15} {'Successes/100':<18} {'Empirical P'}")
    print("-" * 45)
    for n in restart_counts:
        prob = empirical_probs[n]
        print(f"{n:<15} {int(prob * TRIALS):<18} {prob:.4f}")



    # Step 3: Theoretical probabilities P = 1 - (1-p)^n
    print("\n[b.3] Theoretical Probability: P = 1 - (1 - p)^n")
    print(f"  Using p = {p:.4f} (derived above)\n")

    theoretical_probs = {}
    print(f"  {'n':<6} {'Calculation':<35} {'Theoretical P'}")
    print("  " + "-" * 55)
    for n in restart_counts:
        theo = 1 - (1 - p) ** n
        theoretical_probs[n] = theo
        calc_str = f"1 - (1 - {p:.4f})^{n}"
        print(f"  {n:<6} {calc_str:<35} {theo:.4f}")

    # --- Step 4: Comparison table + analysis ---
    print("\n[b.4] Comparison: Empirical vs Theoretical")
    print(f"{'n':<8} {'Empirical P':<15} {'Theoretical P':<15} {'Difference'}")
    print("-" * 50)
    for n in restart_counts:
        emp  = empirical_probs[n]
        theo = theoretical_probs[n]
        diff = abs(emp - theo)
        print(f"{n:<8} {emp:<15.4f} {theo:<15.4f} {diff:.4f}")

In [62]:
main_part_b()

PART (b) - Empirical vs Theoretical Probability of Finding Global Max

[b.1] Deriving p: fraction of starts reaching state 11 via First-Choice HC
------------------------------------------------------------
Start    Terminal     Reached 11?
-----------------------------------
1        2            no
2        2            no
3        2            no
4        4            no
5        4            no
6        4            no
7        7            no
8        7            no
9        7            no
10       7            no
11       11           YES
12       11           YES
13       11           YES
14       11           YES

  States reaching global max: 4 / 14
  p = 4/14 = 0.2857

[b.2] Empirical Probability: 100 independent trials per restart count n
n (restarts)    Successes/100      Empirical P
---------------------------------------------
1               31                 0.3100
3               56                 0.5600
5               83                 0.8300
10              97 

In [63]:
print("""
Explanation:
  p is the probability that a single random restart finds the global max.
  We compute it by running First-Choice HC from every possible start and
  counting how many reach state 11. This p is used in the theoretical formula.
""")


Explanation:
  p is the probability that a single random restart finds the global max.
  We compute it by running First-Choice HC from every possible start and
  counting how many reach state 11. This p is used in the theoretical formula.



In [64]:
print("""
Explanation:
  For each n, we run RRHC 100 times independently and count how often
  state 11 is found. Dividing by 100 gives the empirical probability.
  As n grows, more restarts = more chances to land near state 11.
""")


Explanation:
  For each n, we run RRHC 100 times independently and count how often
  state 11 is found. Dividing by 100 gives the empirical probability.
  As n grows, more restarts = more chances to land near state 11.



In [65]:
print("""
Explanation:
  The formula P = 1-(1-p)^n models the probability of finding the global
  max in at least one of n independent restarts, assuming each restart has
  probability p of success. This is a geometric complement: (1-p)^n is the
  chance ALL restarts fail, so 1 minus that is the chance at least one works.
""")


Explanation:
  The formula P = 1-(1-p)^n models the probability of finding the global
  max in at least one of n independent restarts, assuming each restart has
  probability p of success. This is a geometric complement: (1-p)^n is the
  chance ALL restarts fail, so 1 minus that is the chance at least one works.



In [66]:
print(f"""
Explanation (3-4 sentences):
  The empirical and theoretical probabilities are generally close, especially
  for small n, which validates the independence assumption behind the formula.
  As n increases the theoretical formula predicts higher probabilities more
  optimistically because it assumes each restart is fully independent and p
  is perfectly constant -- but in practice, random starts are not uniformly
  distributed across all 14 states in every trial, causing small deviations.
  Any gap at larger n reveals that the formula's assumption of a fixed p
  breaks down slightly when the landscape has clusters of local maxima that
  bias random starts away from the global max basin of attraction.
""")


Explanation (3-4 sentences):
  The empirical and theoretical probabilities are generally close, especially
  for small n, which validates the independence assumption behind the formula.
  As n increases the theoretical formula predicts higher probabilities more
  optimistically because it assumes each restart is fully independent and p
  is perfectly constant -- but in practice, random starts are not uniformly
  distributed across all 14 states in every trial, causing small deviations.
  Any gap at larger n reveals that the formula's assumption of a fixed p
  breaks down slightly when the landscape has clusters of local maxima that
  bias random starts away from the global max basin of attraction.



In [95]:
#  PART (c) - Plateau Investigation

# Modified landscape: states 7 and 8 both get f = 17 (plateau)
LANDSCAPE_PLATEAU = [5, 8, 6, 12, 9, 7, 17, 17, 10, 6, 19, 15, 11, 8]
                  #  s1  s2  s3  s4  s5  s6  s7  s8  s9 s10 s11 s12 s13 s14


# Now states 7 and 8 both have f=17 -- a plateau just below the global max

In [96]:
def random_restart_hc_plateau(landscape, num_restarts, variant='first_choice',
                               plateau_states=None):
    """
    RRHC with a plateau counter.
    Counts how many restarts terminate ON the plateau instead of global max.
    plateau_states: list of 1-indexed states that form the plateau.
    """
    if plateau_states is None:
        plateau_states = []

    best_state = None
    best_value = -1
    all_results = []
    plateau_terminations = 0

    for _ in range(num_restarts):
        start = random.randint(0, len(landscape) - 1) + 1

        if variant == 'first_choice':
            path, terminal = first_choice_hc(landscape, start)
        else:
            path, terminal = stochastic_hc(landscape, start)

        terminal_value = landscape[terminal - 1]
        all_results.append((start, terminal, path))

        # Count plateau terminations
        if terminal in plateau_states:
            plateau_terminations += 1

        if terminal_value > best_value:
            best_value = terminal_value
            best_state = terminal

    return best_state, best_value, all_results, plateau_terminations

In [97]:
def main_part_c():
    print("PART (c) Plateau Investigation: states 7 and 8 both have f=17")

    print(f"\nOriginal Landscape : {LANDSCAPE}")
    print(f"Modified Landscape : {LANDSCAPE_PLATEAU}")
    print("Change: states 7 and 8 now both have f=17 (plateau region)\n")

    global_max_state = 11
    plateau_states   = [7, 8]   # the plateau
    NUM_RESTARTS     = 20
    TRIALS           = 50       # repeat to get stable discovery rate

    # --- BEFORE plateau modification ---
    print("[c.1] BEFORE plateau -- original landscape, 20 restarts")

    random.seed(300)
    fc_found_before = 0
    for _ in range(TRIALS):
        best_s, _, _, _ = random_restart_hc_plateau(
            LANDSCAPE, NUM_RESTARTS, variant='first_choice', plateau_states=[]
        )
        if best_s == global_max_state:
            fc_found_before += 1

    rate_before = fc_found_before / TRIALS
    print(f"  Global max discovery rate (original landscape): "
          f"{fc_found_before}/{TRIALS} = {rate_before:.2%}")

    # --- AFTER plateau modification ---
    print("\n[c.2] AFTER plateau modification -- states 7 & 8 both have f=17")

    random.seed(300)
    fc_found_after   = 0
    total_plateau_terminations = 0

    print(f"\n  Restart-by-restart detail (one run of 20 restarts shown):")
    print(f"  {'Restart':<10} {'Start':<8} {'Terminal':<12} {'f-value':<10} {'On Plateau?':<14} {'Global Max?'}")

    # Single run for detail
    random.seed(400)
    _, _, detail_results, _ = random_restart_hc_plateau(
        LANDSCAPE_PLATEAU, NUM_RESTARTS, variant='first_choice',
        plateau_states=plateau_states
    )
    for i, (start, terminal, path) in enumerate(detail_results, 1):
        on_plateau  = "YES" if terminal in plateau_states else "no"
        found_global = "YES" if terminal == global_max_state else "no"
        print(f"  {i:<10} {start:<8} {terminal:<12} {LANDSCAPE_PLATEAU[terminal-1]:<10} {on_plateau:<14} {found_global}")

    # 50 trials for stable rate
    random.seed(300)
    for _ in range(TRIALS):
        best_s, _, _, plateau_count = random_restart_hc_plateau(
            LANDSCAPE_PLATEAU, NUM_RESTARTS, variant='first_choice',
            plateau_states=plateau_states
        )
        if best_s == global_max_state:
            fc_found_after += 1
        total_plateau_terminations += plateau_count

    rate_after = fc_found_after / TRIALS
    avg_plateau = total_plateau_terminations / TRIALS

    print(f"\n  Global max discovery rate (plateau landscape): "
          f"{fc_found_after}/{TRIALS} = {rate_after:.2%}")
    print(f"  Average plateau terminations per run of 20 restarts: {avg_plateau:.1f}")

    # Summary table
    print("\n[c.3] Summary Comparison Table")
    print(f"  {'Landscape':<25} {'Discovery Rate':<18} {'Avg Plateau Stuck'}")
    print(f"  {'Original (no plateau)':<25} {rate_before:<18.2%} {'N/A'}")
    print(f"  {'With plateau (s7=s8=17)':<25} {rate_after:<18.2%} {avg_plateau:.1f}/20 restarts")

In [98]:
main_part_c()

PART (c) Plateau Investigation: states 7 and 8 both have f=17

Original Landscape : [5, 8, 6, 12, 9, 7, 17, 14, 10, 6, 19, 15, 11, 8]
Modified Landscape : [5, 8, 6, 12, 9, 7, 17, 17, 10, 6, 19, 15, 11, 8]
Change: states 7 and 8 now both have f=17 (plateau region)

[c.1] BEFORE plateau -- original landscape, 20 restarts
  Global max discovery rate (original landscape): 50/50 = 100.00%

[c.2] AFTER plateau modification -- states 7 & 8 both have f=17

  Restart-by-restart detail (one run of 20 restarts shown):
  Restart    Start    Terminal     f-value    On Plateau?    Global Max?
  1          5        4            12         no             no
  2          9        8            17         YES            no
  3          5        4            12         no             no
  4          13       11           19         no             YES
  5          14       11           19         no             YES
  6          2        2            8          no             no
  7          10       8     

In [100]:
print(f"""
Explanation (3-4 sentences):
  After setting states 7 and 8 to f=17, many restarts that start between
  states 6 and 9 climb up to the plateau at f=17 and stop there, because
  neither neighbour is strictly better (state 6 has f=7, state 9 has f=10,
  and the other plateau state has the same f=17). This directly reduces the
  global max discovery rate compared to the original landscape, because those
  restarts "waste" their climb on the plateau rather than reaching state 11.
  Running more restarts does eventually increase the chance of finding state
  11, but the plateau acts as a persistent trap -- it absorbs restarts that
  start in a wide basin, so the improvement with more restarts is slower than
  in the original landscape without the plateau.
""")



Explanation (3-4 sentences):
  After setting states 7 and 8 to f=17, many restarts that start between
  states 6 and 9 climb up to the plateau at f=17 and stop there, because
  neither neighbour is strictly better (state 6 has f=7, state 9 has f=10,
  and the other plateau state has the same f=17). This directly reduces the
  global max discovery rate compared to the original landscape, because those
  restarts "waste" their climb on the plateau rather than reaching state 11.
  Running more restarts does eventually increase the chance of finding state
  11, but the plateau acts as a persistent trap -- it absorbs restarts that
  start in a wide basin, so the improvement with more restarts is slower than
  in the original landscape without the plateau.



In [101]:
# RUN ALL PARTS

if __name__ == "__main__":
    main_part_a()
    main_part_b()
    main_part_c()

PART (a)  Random Restart HC Implementation

Landscape (14 states): [5, 8, 6, 12, 9, 7, 17, 14, 10, 6, 19, 15, 11, 8]
Global maximum: state 11, f(11) = 19

[a.1] Local Maxima in Q2 Landscape:
---------------------------------------------
  State  2 -> f(2) = 8
  State  4 -> f(4) = 12
  State  7 -> f(7) = 17
  State 11 -> f(11) = 19

  Total local maxima found: 4

[a.2] RRHC with num_restarts=20 -- First-Choice variant
Restart    Start    Terminal     f-value    Global Max Found?
-------------------------------------------------------
1          3        2            8          no
2          8        7            17         no
3          8        7            17         no
4          13       11           19         YES
5          3        2            8          no
6          12       11           19         YES
7          7        7            17         no
8          12       11           19         YES
9          6        4            12         no
10         7        7            17

#Q3

In [102]:
import random

In [106]:
#  QUESTION 3 - Diagnosing & Fixing HC Failures + N-Queens

def get_neighbours(state, landscape):
    """Return left and right neighbours of a state (1-indexed)."""
    neighbours = []
    idx = state - 1
    if idx - 1 >= 0:
        neighbours.append(state - 1)
    if idx + 1 < len(landscape):
        neighbours.append(state + 1)
    return neighbours

In [107]:
def first_choice_hc(landscape, start):
    """First-Choice HC: check left first, move on first strict improvement."""
    current = start
    path = [current]
    while True:
        neighbours = get_neighbours(current, landscape)
        moved = False
        for neighbour in neighbours:
            if landscape[neighbour - 1] > landscape[current - 1]:
                current = neighbour
                path.append(current)
                moved = True
                break
        if not moved:
            break
    return path, current

In [108]:
#  PART (a) - diagnose_hc: detect 3 failure modes

def diagnose_hc(landscape, start):
    """
    Runs First-Choice HC and automatically diagnoses which failure mode occurred.

    Three failure modes:
      1. LOCAL MAXIMUM - no better AND no equal neighbour exists.
                         The algorithm is frozen; every adjacent state is strictly worse.
      2. PLATEAU       - no strictly better neighbour, but at least one equal.
                         Algorithm stops at the flat region edge.
      3. RIDGE         - stuck at a local peak that is NOT the global maximum,
                         AND a valley (lower state) exists between the current position
                         and the global maximum. Reaching the global max requires going
                         downhill first -- which HC strictly forbids.

    Detection logic:
      - PLATEAU  : detected immediately when equal neighbour found, no better.
      - RIDGE    : stuck (no better, no equal) AND current < global max AND
                   a valley exists between current position and global max position.
      - LOCAL MAX: stuck (no better, no equal) AND current == global max
                   (algorithm correctly found the best state and terminates there).

    Note on the relationship between Local Max and Ridge in 1D:
      In a 1D landscape with left/right neighbours, whenever HC is stuck at a
      sub-optimal peak (current < global max), a valley ALWAYS exists between
      that peak and the global max -- because if there were no valley, HC would
      have kept climbing uphill to the global max. Therefore:
        - Sub-optimal stuck state  =>  RIDGE  (valley always present)
        - Stuck at global max      =>  LOCAL MAX  (correct termination)
    """
    current = start
    path    = [current]

    while True:
        neighbours = get_neighbours(current, landscape)
        better = [n for n in neighbours if landscape[n-1] > landscape[current-1]]
        equal  = [n for n in neighbours if landscape[n-1] == landscape[current-1]]

        if better:
            current = better[0]          # First-Choice: leftmost better neighbour
            path.append(current)

        elif equal:
            # Flat neighbour exists, no strictly better -> PLATEAU
            print(f"  Terminated at state {current} with f={landscape[current-1]}."
                  f" Failure mode: PLATEAU")
            return path, current, "plateau"

        else:
            # No better, no equal -> check RIDGE vs LOCAL MAX
            global_max_val = max(landscape)
            current_val    = landscape[current - 1]

            if current_val < global_max_val:
                # Stuck at sub-optimal peak: find global max and check for valley
                global_max_pos = landscape.index(global_max_val) + 1  # 1-indexed
                lo = min(current, global_max_pos)
                hi = max(current, global_max_pos)
                # Valley exists if any state between lo and hi is below BOTH endpoints
                valley = any(
                    landscape[i] < landscape[lo-1] and landscape[i] < landscape[hi-1]
                    for i in range(lo, hi - 1)
                )
                if valley:
                    print(f"  Terminated at state {current} with f={landscape[current-1]}."
                          f" Failure mode: RIDGE")
                    return path, current, "ridge"

            # Stuck at global max (or no valley detected) -> LOCAL MAXIMUM
            print(f"  Terminated at state {current} with f={landscape[current-1]}."
                  f" Failure mode: LOCAL MAXIMUM")
            return path, current, "local_max"

In [109]:
def main_part_a():
    print("PART (a) - diagnose_hc: Detecting Three HC Failure Modes")

    ls_lm = [3, 5, 2, 4, 1]
    print("--- Failure Mode 1: LOCAL MAXIMUM ---")
    print(f"  Landscape : {ls_lm}   (5 states)")
    print(f"  State     :  1   2   3   4   5")
    print(f"  f(s)      :  3   5   2   4   1")
    print(f"  Global max: state 2 (f=5).")
    print(f"  Start = 1 (f=3). Right s2(f=5) is better -> moves to s2.")
    print(f"  At s2(f=5): left=s1(f=3) worse, right=s3(f=2) worse -> stuck.")
    print(f"  f(current)=5 equals global max=5 -> LOCAL MAXIMUM termination.")
    path, term, mode = diagnose_hc(ls_lm, 1)
    print(f"  Path taken  : {path}")
    print(f"  Mode        : {mode.upper()}")

    ls_pl = [4, 8, 12, 12, 12, 9]
    print("--- Failure Mode 2: PLATEAU ---")
    print(f"  Landscape : {ls_pl}   (6 states)")
    print(f"  State     :  1   2   3   4   5   6")
    print(f"  f(s)      :  4   8  12  12  12   9")
    print(f"  States 3, 4, 5 form a flat plateau at f=12.")
    print(f"  Start = 2 (f=8). Right s3(f=12) is better -> moves to s3.")
    print(f"  At s3(f=12): left=s2(f=8) worse, right=s4(f=12) EQUAL -> PLATEAU.")
    path, term, mode = diagnose_hc(ls_pl, 2)
    print(f"  Path taken  : {path}")
    print(f"  Mode        : {mode.upper()}")

    ls_rg = [2, 5, 8, 6, 9, 4, 2]
    print("--- Failure Mode 3: RIDGE ---")
    print(f"  Landscape : {ls_rg}   (7 states)")
    print(f"  State     :  1   2   3   4   5   6   7")
    print(f"  f(s)      :  2   5   8   6   9   4   2")
    print(f"  Two peaks: s3(f=8) and s5(f=9) separated by valley s4(f=6).")
    print(f"  Start = 4 (f=6). Left s3(f=8) is better -> moves to s3.")
    print(f"  At s3(f=8): left=s2(f=5) worse, right=s4(f=6) worse -> stuck.")
    print(f"  global_max=9 at s5. Valley at s4(f=6): 6 < 8 and 6 < 9 -> RIDGE.")
    print(f"  Reaching s5(f=9) requires going DOWNHILL through s4(f=6) -- forbidden.")
    path, term, mode = diagnose_hc(ls_rg, 4)
    print(f"  Path taken  : {path}")
    print(f"  Mode        : {mode.upper()}")

In [110]:
main_part_a()

PART (a) - diagnose_hc: Detecting Three HC Failure Modes
--- Failure Mode 1: LOCAL MAXIMUM ---
  Landscape : [3, 5, 2, 4, 1]   (5 states)
  State     :  1   2   3   4   5
  f(s)      :  3   5   2   4   1
  Global max: state 2 (f=5).
  Start = 1 (f=3). Right s2(f=5) is better -> moves to s2.
  At s2(f=5): left=s1(f=3) worse, right=s3(f=2) worse -> stuck.
  f(current)=5 equals global max=5 -> LOCAL MAXIMUM termination.
  Terminated at state 2 with f=5. Failure mode: LOCAL MAXIMUM
  Path taken  : [1, 2]
  Mode        : LOCAL_MAX
--- Failure Mode 2: PLATEAU ---
  Landscape : [4, 8, 12, 12, 12, 9]   (6 states)
  State     :  1   2   3   4   5   6
  f(s)      :  4   8  12  12  12   9
  States 3, 4, 5 form a flat plateau at f=12.
  Start = 2 (f=8). Right s3(f=12) is better -> moves to s3.
  At s3(f=12): left=s2(f=8) worse, right=s4(f=12) EQUAL -> PLATEAU.
  Terminated at state 3 with f=12. Failure mode: PLATEAU
  Path taken  : [2, 3]
  Mode        : PLATEAU
--- Failure Mode 3: RIDGE ---
  Lan

In [112]:
print("""
  Why HC cannot recover from a Local Maximum:
    At a local maximum every adjacent state has a strictly lower f-value,
    so the uphill condition (f(neighbour) > f(current)) is never satisfied.
    HC has no mechanism to move downhill, so it stays permanently at the peak.
    When the local maximum IS the global maximum the algorithm has succeeded;
    when it is sub-optimal (missed by directional bias), only random restarts
    or a downhill-tolerating algorithm (e.g. Simulated Annealing) can help.
""")


  Why HC cannot recover from a Local Maximum:
    At a local maximum every adjacent state has a strictly lower f-value,
    so the uphill condition (f(neighbour) > f(current)) is never satisfied.
    HC has no mechanism to move downhill, so it stays permanently at the peak.
    When the local maximum IS the global maximum the algorithm has succeeded;
    when it is sub-optimal (missed by directional bias), only random restarts
    or a downhill-tolerating algorithm (e.g. Simulated Annealing) can help.



In [113]:
print("""
  Why HC cannot recover from a Plateau:
    On a plateau the strict uphill condition (f(neighbour) > f(current)) is
    never satisfied because all reachable neighbours have the same f-value.
    The algorithm stops at the first plateau state even though better regions
    may exist beyond the flat area. Sideways moves (with a cap to prevent
    infinite loops) are the only standard fix -- basic HC does not include them.
""")


  Why HC cannot recover from a Plateau:
    On a plateau the strict uphill condition (f(neighbour) > f(current)) is
    never satisfied because all reachable neighbours have the same f-value.
    The algorithm stops at the first plateau state even though better regions
    may exist beyond the flat area. Sideways moves (with a cap to prevent
    infinite loops) are the only standard fix -- basic HC does not include them.



In [114]:
print("""
  Why HC cannot recover from a Ridge:
    A ridge occurs when the global maximum sits across a valley from the
    current local peak. The algorithm identifies that all neighbours of the
    current state are worse, but the only route to the higher peak requires
    moving downhill through the valley first -- which HC strictly forbids.
    The 1D neighbourhood is blind to peaks on the other side of a valley.
    Random restarts are the standard escape: a new random start may land on
    the correct side of the valley and climb directly to the global maximum.
""")


  Why HC cannot recover from a Ridge:
    A ridge occurs when the global maximum sits across a valley from the
    current local peak. The algorithm identifies that all neighbours of the
    current state are worse, but the only route to the higher peak requires
    moving downhill through the valley first -- which HC strictly forbids.
    The 1D neighbourhood is blind to peaks on the other side of a valley.
    Random restarts are the standard escape: a new random start may land on
    the correct side of the valley and climb directly to the global maximum.



In [115]:
#  PART (b) - N-Queens Problem (N=8) with Stochastic HC + RRHC

def count_conflicts(board):
    """
    Count the number of attacking pairs of queens.
    board[i] = row of queen in column i.
    Attacks: same row (board[i]==board[j]) or same diagonal (|row diff|==|col diff|).
    Lower is better; 0 = a valid solution.
    """
    conflicts = 0
    n = len(board)
    for i in range(n):
        for j in range(i + 1, n):
            if board[i] == board[j]:                        # same row
                conflicts += 1
            elif abs(board[i] - board[j]) == abs(i - j):   # same diagonal
                conflicts += 1
    return conflicts

In [116]:
def stochastic_hc_nqueens(board):
    """
    Stochastic HC for N-Queens using a COLUMN-BASED neighbourhood.

    At each step:
      1. For every column c and every possible row r (0..N-1):
         try placing the queen in column c at row r.
      2. Collect ALL such moves that strictly reduce the conflict count.
      3. If any improving moves exist, pick one RANDOMLY (stochastic uphill).
      4. If no improving move exists, stop (local minimum reached).

    This column-based neighbourhood (N*N = 64 possible moves for N=8) is
    much richer than pairwise swaps, dramatically reducing local minima traps.
    """
    current           = board[:]
    current_conflicts = count_conflicts(current)
    n                 = len(current)

    while current_conflicts > 0:
        improving = []
        for col in range(n):
            for row in range(n):
                if row == current[col]:
                    continue                            # skip unchanged position
                nb = current[:]
                nb[col] = row                          # try placing queen at new row
                nc = count_conflicts(nb)
                if nc < current_conflicts:
                    improving.append((nb, nc))

        if not improving:
            break                                      # local minimum -- no improving move

        chosen, chosen_c = random.choice(improving)   # stochastic: pick randomly
        current           = chosen
        current_conflicts = chosen_c

    return current, current_conflicts

In [117]:
def random_board(n=8):
    """Generate a random board: one queen per column, random row 0..N-1."""
    return [random.randint(0, n - 1) for _ in range(n)]

In [118]:
def solve_nqueens_rrhc(num_restarts, n=8):
    """
    Random Restart HC for N-Queens.
    Tries up to num_restarts random starting boards.
    Stops early if a 0-conflict solution is found.
    Returns (best_board, best_conflicts, restarts_used).
    """
    best_board     = None
    best_conflicts = float('inf')

    for restart in range(1, num_restarts + 1):
        board, conflicts = stochastic_hc_nqueens(random_board(n))

        if conflicts < best_conflicts:
            best_conflicts = conflicts
            best_board     = board[:]

        if conflicts == 0:
            return best_board, 0, restart   # solution found

    return best_board, best_conflicts, num_restarts

In [119]:
def print_board(board):
    """Print N-Queens board using Q (queen) and . (empty) characters."""
    n = len(board)
    print("  +" + "---+" * n)
    for row in range(n):
        line = "  |"
        for col in range(n):
            line += " Q |" if board[col] == row else " . |"
        print(line)
        print("  +" + "---+" * n)
    print("  " + "".join(f"  {c+1} " for c in range(n)))

In [123]:
def main_part_b():
    print("PART (b) - N-Queens (N=8): Stochastic HC + Random Restarts")

    print("""
Representation : board[i] = row of queen in column i  (list of 8 integers, rows 0-7).
Fitness        : count_conflicts(board) = attacking pairs (lower is better; 0 = solved).
Neighbourhood  : for each column, try all N possible row placements = N*N = 64 moves.
Move           : from all improving moves, pick one RANDOMLY (stochastic uphill).
Restart        : generate a fresh random board and re-run HC from scratch.
""")

    # --- Demonstrate count_conflicts ---
    print("[b.1] Demonstrating count_conflicts on known boards:")
    print("-" * 58)
    demos = [
        ([0, 1, 2, 3, 4, 5, 6, 7], "all on main diagonal -- 28 attacking pairs"),
        ([0, 4, 7, 5, 2, 6, 1, 3], "known 8-Queens solution"),
        ([3, 6, 2, 7, 1, 4, 0, 5], "another known 8-Queens solution"),
        ([0, 0, 0, 0, 0, 0, 0, 0], "all queens in row 0 -- 28 row conflicts"),
    ]
    for b, desc in demos:
        c = count_conflicts(b)
        tag = "  <-- SOLUTION!" if c == 0 else ""
        print(f"  Board : {b}")
        print(f"  Conflicts = {c}{tag}   ({desc})")
        print()

    # --- Run solve_nqueens_rrhc(100) ---
    print("[b.2] Running solve_nqueens_rrhc(100):")
    random.seed(42)
    solution, conflicts, restarts_used = solve_nqueens_rrhc(100, n=8)

    print(f"  Restarts needed  : {restarts_used}")
    print(f"  Final conflicts  : {conflicts}")
    print(f"  Final board      : {solution}")
    if conflicts == 0:
        print(f"  Result           : SOLUTION FOUND -- 0 conflicts!")
    else:
        print(f"  Result           : Best found -- {conflicts} conflict(s) remain")

    print("\n  Visual board (row 0 = top, columns 1-8 = left to right):")
    print_board(solution)

In [124]:
main_part_b()

PART (b) - N-Queens (N=8): Stochastic HC + Random Restarts

Representation : board[i] = row of queen in column i  (list of 8 integers, rows 0-7).
Fitness        : count_conflicts(board) = attacking pairs (lower is better; 0 = solved).
Neighbourhood  : for each column, try all N possible row placements = N*N = 64 moves.
Move           : from all improving moves, pick one RANDOMLY (stochastic uphill).
Restart        : generate a fresh random board and re-run HC from scratch.

[b.1] Demonstrating count_conflicts on known boards:
----------------------------------------------------------
  Board : [0, 1, 2, 3, 4, 5, 6, 7]
  Conflicts = 28   (all on main diagonal -- 28 attacking pairs)

  Board : [0, 4, 7, 5, 2, 6, 1, 3]
  Conflicts = 0  <-- SOLUTION!   (known 8-Queens solution)

  Board : [3, 6, 2, 7, 1, 4, 0, 5]
  Conflicts = 0  <-- SOLUTION!   (another known 8-Queens solution)

  Board : [0, 0, 0, 0, 0, 0, 0, 0]
  Conflicts = 28   (all queens in row 0 -- 28 row conflicts)

[b.2] Running 

In [125]:
print("""
Explanation:
  Each column holds exactly one queen (guaranteed by the list representation).
  The column-based neighbourhood (N*N=64 moves) evaluates every possible row
  placement for every column at each step, dramatically reducing local minima
  compared to the pairwise-swap approach (only 28 possible moves).
  Stochastic selection (random.choice among all improving moves) avoids the
  deterministic bias of First-Choice, exploring more diverse paths through
  the conflict landscape.
  Random restarts escape local minima by generating a completely fresh random
  board whenever no improving move is available, allowing the search to try
  different regions of the conflict landscape.
""")


Explanation:
  Each column holds exactly one queen (guaranteed by the list representation).
  The column-based neighbourhood (N*N=64 moves) evaluates every possible row
  placement for every column at each step, dramatically reducing local minima
  compared to the pairwise-swap approach (only 28 possible moves).
  Stochastic selection (random.choice among all improving moves) avoids the
  deterministic bias of First-Choice, exploring more diverse paths through
  the conflict landscape.
  Random restarts escape local minima by generating a completely fresh random
  board whenever no improving move is available, allowing the search to try
  different regions of the conflict landscape.



In [127]:
#  PART (c) - Benchmark N-Queens Solver

def main_part_c():
    print("PART (c) - Benchmarking solve_nqueens_rrhc(k)  [N=8, 30 trials each]")

    k_values = [5, 10, 25, 50, 100]
    TRIALS   = 30
    n        = 8

    print(f"\nk values tested : {k_values}")
    print(f"Trials per k    : {TRIALS}")
    print(f"N (queens)      : {n}")
    print(f"Neighbourhood   : column-based (N*N = 64 moves per step)\n")

    all_results = {}

    for k in k_values:
        successes      = 0
        restart_counts = []

        for trial in range(TRIALS):
            random.seed(trial * 300 + k * 7)
            board, conflicts, used = solve_nqueens_rrhc(k, n=n)
            if conflicts == 0:
                successes += 1
                restart_counts.append(used)

        sr    = successes / TRIALS
        avg_r = (sum(restart_counts) / len(restart_counts)) if restart_counts else 0
        all_results[k] = (sr, avg_r, successes)

    print(f"{'k':<8} {'Successes/30':<16} {'Success Rate':<16} {'Avg Restarts to Solution'}")
    print("-" * 65)
    for k in k_values:
        sr, avg_r, succ = all_results[k]
        avg_str = f"{avg_r:.1f}" if avg_r > 0 else "N/A"
        print(f"{k:<8} {str(succ)+'/30':<16} {sr:<16.2%} {avg_str}")

In [128]:
main_part_c()

PART (c) - Benchmarking solve_nqueens_rrhc(k)  [N=8, 30 trials each]

k values tested : [5, 10, 25, 50, 100]
Trials per k    : 30
N (queens)      : 8
Neighbourhood   : column-based (N*N = 64 moves per step)

k        Successes/30     Success Rate     Avg Restarts to Solution
-----------------------------------------------------------------
5        17/30            56.67%           2.5
10       25/30            83.33%           4.2
25       28/30            93.33%           7.3
50       30/30            100.00%          7.6
100      30/30            100.00%          6.7


In [129]:
print("""
Analysis (5-6 sentences):

  Reason 1 -- Deceptive local minima in the conflict landscape:
  The N-Queens conflict landscape contains many local minima where every
  possible column-row placement either keeps the conflict count the same
  or raises it, yet the board still has 1-2 conflicts remaining. These
  dead-end states arise because fixing the last conflicting pair by moving
  one queen inevitably introduces new conflicts between previously non-attacking
  queens -- a structural property of the 8x8 conflict geometry that traps
  stochastic HC regardless of how many improving steps preceded the trap.

  Reason 2 -- Shrinking improving neighbourhood near solutions:
  As conflicts reduce from the initial count toward 1, the fraction of
  available improving moves (out of 64 column-row combinations) shrinks
  dramatically. Near a solution almost all moves are neutral or harmful --
  the algorithm navigates an increasingly narrow funnel where the exact move
  needed to eliminate the last conflict is rarely available in the current
  board's neighbourhood, making the final step from 1 conflict to 0 the
  hardest of all.

  Experimental confirmation:
  The benchmark table shows success rate rising clearly with k: from 56.67%
  at k=5 to 100% at k=50 and k=100, directly confirming that more restarts
  overcome local minima by providing fresh random starting boards. The average
  restarts-to-success (2.5 at k=5, 7.6 at k=50) is well below k in all cases,
  showing that most successful trials find a solution quickly -- it is the
  rare unlucky board configurations that demand the larger restart budget.

  Remaining limitation for N=1000:
  For N=1000 the search space is 1000^1000 possible boards and each HC step
  must evaluate N*N=1,000,000 column-row moves, making each restart extremely
  slow. More critically, the probability of a random board lying in the basin
  of attraction of a solution shrinks exponentially with N, so even with
  unlimited restarts RRHC cannot reliably solve N=1000 in feasible time.
  The MIN-CONFLICTS algorithm is far more suitable: it directly targets the
  most conflicted queen and moves it to the row that minimises its conflicts,
  solving N=1,000,000 in near-linear time by exploiting domain-specific
  structure that RRHC completely lacks.
""")


Analysis (5-6 sentences):

  Reason 1 -- Deceptive local minima in the conflict landscape:
  The N-Queens conflict landscape contains many local minima where every
  possible column-row placement either keeps the conflict count the same
  or raises it, yet the board still has 1-2 conflicts remaining. These
  dead-end states arise because fixing the last conflicting pair by moving
  one queen inevitably introduces new conflicts between previously non-attacking
  queens -- a structural property of the 8x8 conflict geometry that traps
  stochastic HC regardless of how many improving steps preceded the trap.

  Reason 2 -- Shrinking improving neighbourhood near solutions:
  As conflicts reduce from the initial count toward 1, the fraction of
  available improving moves (out of 64 column-row combinations) shrinks
  dramatically. Near a solution almost all moves are neutral or harmful --
  the algorithm navigates an increasingly narrow funnel where the exact move
  needed to eliminate the l

In [130]:
#  RUN ALL PARTS

if __name__ == "__main__":
    main_part_a()
    main_part_b()
    main_part_c()

PART (a) - diagnose_hc: Detecting Three HC Failure Modes
--- Failure Mode 1: LOCAL MAXIMUM ---
  Landscape : [3, 5, 2, 4, 1]   (5 states)
  State     :  1   2   3   4   5
  f(s)      :  3   5   2   4   1
  Global max: state 2 (f=5).
  Start = 1 (f=3). Right s2(f=5) is better -> moves to s2.
  At s2(f=5): left=s1(f=3) worse, right=s3(f=2) worse -> stuck.
  f(current)=5 equals global max=5 -> LOCAL MAXIMUM termination.
  Terminated at state 2 with f=5. Failure mode: LOCAL MAXIMUM
  Path taken  : [1, 2]
  Mode        : LOCAL_MAX
--- Failure Mode 2: PLATEAU ---
  Landscape : [4, 8, 12, 12, 12, 9]   (6 states)
  State     :  1   2   3   4   5   6
  f(s)      :  4   8  12  12  12   9
  States 3, 4, 5 form a flat plateau at f=12.
  Start = 2 (f=8). Right s3(f=12) is better -> moves to s3.
  At s3(f=12): left=s2(f=8) worse, right=s4(f=12) EQUAL -> PLATEAU.
  Terminated at state 3 with f=12. Failure mode: PLATEAU
  Path taken  : [2, 3]
  Mode        : PLATEAU
--- Failure Mode 3: RIDGE ---
  Lan